# tiny-voca 4.5M — English TTS on a free T4 or CPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/tiny-voca-4.5M-english-tts.ipynb)

[`n4ze3m/tiny-voca`](https://huggingface.co/n4ze3m/tiny-voca) is a small **English** text-to-speech model with two voices, **F2** (female) and **M2** (male). The complete model, including the waveform generator, has **4,537,291** parameters (`tiny-voca.pt` **18.2 MB**, PyTorch float32). Output is **24 kHz** mono, American English. It was trained from scratch on about **18 hours** of synthetic speech. [Apache-2.0](https://huggingface.co/n4ze3m/tiny-voca). The [card](https://huggingface.co/n4ze3m/tiny-voca) says it runs about **70× faster than real time** on four CPU threads.

**Why this model.** Two built-in voices in an 18 MB checkpoint that the card also ships as ONNX (`onnx/model.onnx`, 21.4 MB). Text is converted to phonemes with [`dizione`](https://github.com/n4ze3m/dizione) — no eSpeak, no voice cloning, no extra vocoder download. GGUF files live on the Hub too; this notebook uses the PyTorch and ONNX paths.

This is the free-Colab tiny English TTS notebook with two fixed voices. [Phonon 31M](phonon-31m-en-streaming-tts.ipynb) is a slightly larger English streaming model. [Silencio-2](Silencio-2-tts-voice-clone.ipynb) adds tags and cloning. [ZONOS3-Mini](ZONOS3-Mini-voice-clone-tts.ipynb) is multilingual with cloning. [Drama 3](Drama-3-tts-voice-clone.ipynb) needs a Colab Pro A100.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/tiny-voca-4.5M-english-tts.md) for when this notebook is a good fit and when it is not.

**Licence:** [Apache-2.0](https://huggingface.co/n4ze3m/tiny-voca) on the weights and code. Files in `phonemizer/` are derived from the CMU Pronouncing Dictionary and other sources; their licenses are in `phonemizer/LICENSE.cmudict` and `phonemizer/LICENSE.homograph-data`.

**Runtime:** CPU is enough. *Runtime → Change runtime type → T4 GPU* if you have it — the PyTorch path uses CUDA when it is available; the ONNX demo stays on CPU. First run downloads `tiny-voca.pt` (18.2 MB), `onnx/model.onnx` (21.4 MB), `config.json`, and `phonemizer/` (about 5.4 MB). No Hugging Face token. Peak VRAM is printed if CUDA is used.

**Not in this notebook:** GGUF / llama.cpp, the browser Space, training, or voice cloning (the card has two fixed voices only).

**Links:** [model card](https://huggingface.co/n4ze3m/tiny-voca) · [code](https://github.com/n4ze3m/tiny-voca) · [dizione](https://github.com/n4ze3m/dizione) · [Space](https://huggingface.co/spaces/n4ze3m/tiny-voca-studio) · [announcement](https://x.com/n4ze3m/status/2121799813679419515) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/tiny-voca-4.5M-english-tts.md)


## 0. Early runtime check

Print `torch.cuda.is_available()`, the device name, and free VRAM before installing anything. CPU is enough. A T4 is optional.


In [ ]:
import os
import shutil

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch

print("torch.cuda.is_available():", torch.cuda.is_available())
print("torch", torch.__version__, "| cuda", torch.version.cuda)

if torch.cuda.is_available():
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability()
    print("GPU name:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
    print(f"VRAM free / total: {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB")
else:
    print("GPU name: none")
    print("VRAM free: n/a")
    print("CPU is enough for this notebook.")

print("nvidia-smi:", shutil.which("nvidia-smi") or "not on PATH")


## 1. Install dizione and tiny-voca (keep Colab's torch)

The [model card](https://huggingface.co/n4ze3m/tiny-voca#usage) install is:

```bash
pip install git+https://github.com/n4ze3m/dizione git+https://github.com/n4ze3m/tiny-voca
hf download n4ze3m/tiny-voca --local-dir tiny-voca
export DIZIONE_DATA=tiny-voca/phonemizer
```

Current Colab already has a CUDA `torch`. This cell installs dizione, tiny-voca, and CPU `onnxruntime` and does **not** replace Colab's torch. Weights are downloaded in the next cell.

Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`.


In [ ]:
%pip install -q git+https://github.com/n4ze3m/dizione git+https://github.com/n4ze3m/tiny-voca onnxruntime

# Card: pip install git+https://github.com/n4ze3m/dizione git+https://github.com/n4ze3m/tiny-voca
# Keep Colab's CUDA torch. Use %pip, not python -I. onnxruntime is the CPU wheel for demo 4.

import sys

import torch

print("python", sys.version.split()[0])
print("torch (keep Colab's wheel)", torch.__version__, "| cuda", torch.version.cuda)

import dizione
import tiny_voca
import onnxruntime as ort

print("dizione", dizione.__version__)
print("tiny_voca", getattr(tiny_voca, "__version__", "ok"))
print("onnxruntime", ort.__version__)


> Restart is usually unnecessary: this install does not upgrade `transformers` or replace torch. If a later cell cannot import `dizione` or `tiny_voca`, use *Runtime → Restart session* and continue from the next cell. You do not need to rerun the install unless it failed.


## 2. Download the checkpoint

Same Hub repo as `hf download n4ze3m/tiny-voca --local-dir tiny-voca`. This cell skips GGUF and sample WAVs; it keeps `tiny-voca.pt`, `onnx/model.onnx`, `config.json`, and `phonemizer/` (dizione data). Then `DIZIONE_DATA` is pointed at that folder, as the card requires.


In [ ]:
import os
from pathlib import Path

from huggingface_hub import snapshot_download

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

MODEL_ID = "n4ze3m/tiny-voca"
MODEL_DIR = Path("tiny-voca")

if MODEL_ID != "n4ze3m/tiny-voca":
    raise SystemExit("This notebook only loads n4ze3m/tiny-voca.")

snapshot_download(
    MODEL_ID,
    local_dir=str(MODEL_DIR),
    token=False,
    allow_patterns=[
        "tiny-voca.pt",
        "onnx/model.onnx",
        "config.json",
        "phonemizer/**",
    ],
)

os.environ["DIZIONE_DATA"] = str((MODEL_DIR / "phonemizer").resolve())

CHECKPOINT = MODEL_DIR / "tiny-voca.pt"
ONNX_PATH = MODEL_DIR / "onnx" / "model.onnx"
for path in (CHECKPOINT, ONNX_PATH, MODEL_DIR / "phonemizer" / "lexicon.tsv"):
    if not path.is_file():
        raise SystemExit(f"missing {path}")

print("DIZIONE_DATA", os.environ["DIZIONE_DATA"])
for rel in ("tiny-voca.pt", "onnx/model.onnx", "config.json"):
    p = MODEL_DIR / rel
    print(f"{rel:20s} {p.stat().st_size / 1e6:.1f} MB")
print(
    "phonemizer/",
    round(sum(p.stat().st_size for p in (MODEL_DIR / "phonemizer").rglob("*") if p.is_file()) / 1e6, 1),
    "MB",
)


## 3. Load the PyTorch model

`tiny_voca.infer.match_frontend` selects dizione (`own`) from the checkpoint fingerprint, then `load_model` reads `tiny-voca.pt`. CUDA is used when it is available; otherwise CPU. The card CLI is `python -m tiny_voca.infer --checkpoint tiny-voca/tiny-voca.pt --voice F2 --text ... --output hello.wav`, with `--speed`, `--pitch-shift`, `--pitch-range`, and `--text-file`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

from tiny_voca.infer import load_model, match_frontend, speak_text
from tiny_voca.text import frontend as current_frontend

os.environ.setdefault("DIZIONE_DATA", str(Path("tiny-voca/phonemizer").resolve()))
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

CHECKPOINT = Path("tiny-voca/tiny-voca.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

matched = match_frontend(CHECKPOINT)
if matched != "own":
    raise SystemExit(f"expected dizione frontend 'own', got {matched!r}. Is DIZIONE_DATA set?")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

model = load_model(CHECKPOINT, device=DEVICE)
n_params = sum(p.numel() for p in model.parameters())
SAMPLE_RATE = model.config.sample_rate

print("frontend", matched, "| tiny_voca.text", current_frontend())
print("device", next(model.parameters()).device)
print("params", f"{n_params:,}", "| voices", list(model.config.speakers))
print("sample_rate", SAMPLE_RATE, "Hz | max_tokens", model.config.max_tokens)

if torch.cuda.is_available():
    print(
        f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**2:.1f} MB | "
        f"peak: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB"
    )


def play_wave(wave, name):
    audio = np.asarray(wave.detach().cpu().numpy(), dtype=np.float32)
    path = f"{name}.wav"
    sf.write(path, audio, SAMPLE_RATE)
    print(f"{path}: {len(audio) / SAMPLE_RATE:.2f}s | peak={float(np.abs(audio).max()):.3f}")
    display(Audio(audio, rate=SAMPLE_RATE))
    return path


## 4. Both voices on the same line

F2 is female, M2 is male. The line is a sample from the [model card](https://huggingface.co/n4ze3m/tiny-voca#samples).


In [ ]:
LINE = (
    "Hello! This is tiny voca, a small text to speech model that runs on a phone or in a browser."
)

for voice in ("F2", "M2"):
    print("voice", voice)
    wave, spoken = speak_text(model, LINE, voice)
    print("utterances", len(spoken), "| phones", " ".join(spoken[0].symbols[:12]), "...")
    play_wave(wave, f"tiny-voca-{voice}")


## 5. Speed, pitch-shift, and pitch-range

Same options as the card CLI. `--speed` is speaking rate (1.0 default). `--pitch-shift` is added to log F0; **0.058 is about one semitone**. `--pitch-range` scales pitch movement around the utterance mean (1.0 as predicted).


In [ ]:
CONTROL_LINE = "Did you really think I would forget your birthday?"

variants = [
    ("default", dict(speed=1.0, pitch_shift=0.0, pitch_range=1.0)),
    ("speed 1.25", dict(speed=1.25, pitch_shift=0.0, pitch_range=1.0)),
    ("pitch-shift +1 semitone", dict(speed=1.0, pitch_shift=0.058, pitch_range=1.0)),
    ("pitch-range 1.6", dict(speed=1.0, pitch_shift=0.0, pitch_range=1.6)),
]

print("text:", CONTROL_LINE)
print("voice: F2")
for label, kwargs in variants:
    print(label, kwargs)
    wave, _ = speak_text(model, CONTROL_LINE, "F2", **kwargs)
    play_wave(wave, "tiny-voca-ctrl-" + label.replace(" ", "-").replace("+", ""))


## 6. Long paragraph: wall time and real-time factor

`speak_text` splits on sentences and joins the audio (the card: text of any length works). Real-time factor here is **audio seconds / wall seconds** (higher is faster). The card's published figure is about **70×** on four CPU threads; the numbers below are this runtime, after a one-line warmup. CPU is always measured. GPU is measured too when CUDA is available.


In [ ]:
import time

PARAGRAPH = (
    "The meeting moved to 3:30 pm on March 14th, so we have about 45 minutes left. "
    "She had never seen the sea before, and for a long while she stood at the edge "
    "of the water without saying a word. "
    "Did you really think I would forget your birthday? "
    "Hello! This is tiny voca, a small text to speech model that runs on a phone or in a browser."
)


def time_speak(text, voice, device):
    moved = model.to(device)
    speak_text(moved, "Ready.", voice)
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    wave, spoken = speak_text(moved, text, voice)
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    audio_s = float(wave.numel()) / SAMPLE_RATE
    rtf = audio_s / elapsed if elapsed > 0 else float("inf")
    print(
        f"{device}: wall {elapsed:.3f}s | audio {audio_s:.2f}s | "
        f"{rtf:.1f}x realtime | utterances {len(spoken)}"
    )
    if device == "cuda":
        print(
            f"  VRAM allocated {torch.cuda.memory_allocated() / 1024**2:.1f} MB | "
            f"peak {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB"
        )
    return wave


print("paragraph voices timed on F2")
print(PARAGRAPH)
print()
wave_cpu = time_speak(PARAGRAPH, "F2", "cpu")
play_wave(wave_cpu, "tiny-voca-paragraph-cpu")

if torch.cuda.is_available():
    print()
    wave_gpu = time_speak(PARAGRAPH, "F2", "cuda")
    play_wave(wave_gpu, "tiny-voca-paragraph-gpu")
else:
    print("GPU: none — CPU figure above is the one to compare with the card's ~70× on four threads.")


## 7. ONNX Runtime on CPU

The [card's ONNX path](https://huggingface.co/n4ze3m/tiny-voca#onnx): `dizione.phonemize` → phoneme ids and stress, then an `onnxruntime` session. Inputs are `tokens`, `stress`, `voice` (0 = F2, 1 = M2), `speed`, `pitch_shift`, `pitch_range`. The graph returns a 24 kHz waveform; scale the peak to 0.99 before saving. One utterance per call, up to 512 tokens and 32 seconds — split longer text into sentences (`tiny_voca.infer.speak_text` does that in Python). End each utterance with a mark such as a full stop.


In [ ]:
import os
import time
from pathlib import Path

import dizione
import numpy as np
import onnxruntime as ort
import soundfile as sf
import torch
from IPython.display import Audio, display

os.environ.setdefault("DIZIONE_DATA", str(Path("tiny-voca/phonemizer").resolve()))
dizione.reset()

ONNX_TEXT = "Hello! This is tiny voca."
phones = dizione.phonemize(ONNX_TEXT)
one = lambda value: np.array([value], dtype=np.float32)

session = ort.InferenceSession(
    "tiny-voca/onnx/model.onnx",
    providers=["CPUExecutionProvider"],
)
print("providers", session.get_providers())
print("inputs", [item.name for item in session.get_inputs()])
print("phonemes", len(phones.ids), "ids")

t0 = time.perf_counter()
audio, durations = session.run(
    None,
    {
        "tokens": np.array([phones.ids], dtype=np.int64),
        "stress": np.array([phones.stress], dtype=np.int64),
        "voice": np.array([0], dtype=np.int64),  # 0 = F2, 1 = M2
        "speed": one(1.0),
        "pitch_shift": one(0.0),
        "pitch_range": one(1.0),
    },
)
elapsed = time.perf_counter() - t0
audio = np.asarray(audio[0], dtype=np.float32)
audio = audio * min(1.0, 0.99 / np.abs(audio).max())
audio_s = audio.shape[-1] / 24000
print(
    f"ONNX CPU: wall {elapsed:.3f}s | audio {audio_s:.2f}s | "
    f"{audio_s / elapsed:.1f}x realtime | peak={float(np.abs(audio).max()):.3f}"
)
print("durations", durations.shape, "(frames of 256 samples)")

sf.write("tiny-voca-onnx-F2.wav", audio, 24000)
display(Audio(audio, rate=24000))

if torch.cuda.is_available():
    print(
        f"VRAM after ONNX (CPU session): allocated {torch.cuda.memory_allocated() / 1024**2:.1f} MB | "
        f"peak {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB"
    )


## 8. Caveats and cleanup

- **Licence.** Apache-2.0 on the weights and code. Keep the `phonemizer/` licenses (`LICENSE.cmudict`, `LICENSE.homograph-data`).
- **English only, two fixed voices.** F2 and M2. No voice cloning, no streaming, no other languages. Foreign names and unusual spellings are often mispronounced; numbers, abbreviations, and homographs depend on dizione.
- **Synthetic training data.** About 18 hours of synthetic speech. The card: the voice is clear but can sound slightly synthetic; the F2 voice sounds younger than intended; the same text is always read the same way; speech is slightly faster than the training recordings at the default speed.
- **Install.** Card: `pip install git+https://github.com/n4ze3m/dizione git+https://github.com/n4ze3m/tiny-voca`. Keep Colab's torch. Use `%pip`, not `python -I`. Set `DIZIONE_DATA` to `tiny-voca/phonemizer`.
- **ONNX.** One utterance per call (512 tokens / 32 seconds). End with a mark. Scale the peak to 0.99. This notebook forces `CPUExecutionProvider`.
- **Not Phonon, not Silencio-2, not ZONOS3-Mini, not Drama 3.** Phonon 31M streams English from a 64.5 MB q8 file. Silencio-2 adds tags and cloning. ZONOS3-Mini is multilingual with cloning. Drama 3 needs a Pro A100.
- **Responsible use.** Do not impersonate a real person or deceive listeners about who is speaking. Label synthetic speech as synthetic where that matters.

Free the GPU before you open another notebook. Peak VRAM below is `torch.cuda.max_memory_allocated` for this session.


In [ ]:
import gc

if torch.cuda.is_available():
    print(f"before cleanup: allocated {torch.cuda.memory_allocated() / 1024**2:.1f} MB")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**2:.1f} MB")

for name in ("model", "session", "wave", "wave_cpu", "wave_gpu", "audio", "phones"):
    globals().pop(name, None)
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(
        f"after cleanup: allocated {torch.cuda.memory_allocated() / 1024**2:.1f} MB | "
        f"free / total {free_b / 1024**3:.1f} / {total_b / 1024**3:.1f} GB"
    )
else:
    print("CPU runtime — nothing to empty on the GPU.")
